## **Advance RAG Model Trained on Attached Knowledge Base**

- knowledge-base/: This directory is where our raw data lives. It contains a text_data/ folder full of .txt and .txt.clean files. These are the documents that act as the "brain" or context for our assistant.
- vector_db/: This directory contains a SQLite database (chroma.sqlite3). This is our Chroma vector database, where all the text from the knowledge-base is stored as numerical vectors (embeddings) for fast search.
- RAG.ipynb: This is your main script tying everything together. It reads the files, chunks them, stores them in the database, and provides a Chat Interface.

### **Importing Libraries**

In [22]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from huggingface_hub import login
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import gradio as gr

### **SetUp & Authentication**

In [23]:
load_dotenv(override=True)
db_name = "vector_db"

openai = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.getenv("Groq_API_KEY"))
MODEL= "llama-3.3-70b-versatile"

In [24]:
# Log in to HuggingFace

hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### **Load Our Documents From /knowledge-base**

- It looks inside the knowledge-base/text_data folder for text files. It uses a custom SafeTextLoader to prevent the code from crashing if it encounters weird text encodings. It also tags every document with custom metadata (e.g., specifying if it was a .clean file or a raw file).

In [25]:
# 1. Updated SafeTextLoader to handle lazy_load
class SafeTextLoader(TextLoader):
    """
    Custom loader that tries UTF-8 first, then falls back to Latin-1 
    if a decoding error occurs.
    """
    def lazy_load(self):
        try:
            # 1. Try UTF-8 first
            self.encoding = "utf-8"
            # We must iterate over the generator to trigger the read
            for doc in super().lazy_load():
                yield doc
        except RuntimeError:
            # TextLoader raises RuntimeError when encoding fails.
            # 2. Fallback to Latin-1 (which reads almost any byte stream)
            self.encoding = "latin-1"
            for doc in super().lazy_load():
                yield doc

base_path = "knowledge-base/text_data"
documents = []
glob_patterns = ["**/*.txt", "**/*.txt.clean"]

for pattern in glob_patterns:
    loader = DirectoryLoader(base_path, glob=pattern, loader_cls=SafeTextLoader, show_progress=True)
    
    try:
        loaded_docs = loader.load()
        
        for doc in loaded_docs:
            full_path = doc.metadata.get("source", "")
            filename = os.path.basename(full_path)
            
            # Apply your metadata logic
            doc.metadata["source_file"] = filename
            doc.metadata["source_type"] = "clean" if filename.endswith(".txt.clean") else "raw"
            
            documents.append(doc)
            
    except Exception as e:
        print(f"Error processing pattern {pattern}: {e}")

print(f"Loaded {len(documents)} documents")

100%|██████████| 150/150 [00:00<00:00, 3919.01it/s]

Loaded 165 documents


- Our all Documents are Loaded into 'documents' list.

In [26]:
documents[18]

Document(metadata={'source': 'knowledge-base\\text_data\\S08_set1_a3.txt.clean', 'source_file': 'S08_set1_a3.txt.clean', 'source_type': 'clean'}, page_content='penguin\n\n\n\nA penguin encounters a human during Antarctic summer.\nPenguins (order Sphenisciformes, family Spheniscidae) are a group of aquatic, flightless birds living almost exclusively in the Southern Hemisphere.\n\nThe number of penguin species is debated. Depending on which authority is followed, penguin biodiversity varies between 17 and 20 living species, all in the subfamily Spheniscinae. Some sources consider the White-flippered Penguin a separate Eudyptula species, while others treat it as a subspecies of the Little Penguin (e.g. Williams, 1995; Davis & Renner, 2003); the actual situation seems to be more complicated (Banks et al. 2002). Similarly,  it is still unclear whether the Royal Penguin is merely a color morph of the Macaroni penguin. Also eligible to be a separate species is the Northern population of Rockh

### **Text Splitting (Chunking)**

- LLMs have context limits, so we can't feed them an entire book at once. This splits our large documents into smaller chunks of up to 500 characters, with 150 characters of overlap between chunks so that sentences don't get abruptly cut off and lose context.

In [27]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 13697 chunks
First chunk:

page_content='Kangaroo
Leopard
Penguin
Polar_bear
Elephant
Wolf
Otter
Beetle
Turtle
Duck' metadata={'source': 'knowledge-base\\text_data\\S08_set1_topics.txt', 'source_file': 'S08_set1_topics.txt', 'source_type': 'raw'}


- Here we Divided all our 165 Documents into 13697 chunks as shown above

In [28]:
chunks[10000]

Document(metadata={'source': 'knowledge-base\\text_data\\S10_set3_a10.txt.clean', 'source_file': 'S10_set3_a10.txt.clean', 'source_type': 'clean'}, page_content='and designed in the regular style. It is situated on the southern bank of the Neva at the head of the Fontanka and is famous for its cast iron railing and marble sculptures.')

### **Generating Embeddings & Storing in Vector DB**

- Here we downloads the 'all-MiniLM-L6-v2' embedding model from HuggingFace which converts every single chunk of text into a mathematical vector and saves all of them into the 'vector_db' folder using Chroma. This allows the app to find text similar to the user's question mathematically.

In [11]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 13697 documents


In [29]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 13,697 vectors with 384 dimensions in the vector store


In [30]:
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

### **Chat LLM and Retriever Setup**

- Here we turns our Chroma database into a "retriever" (a search engine). It also initializes ChatOpenAI, again tricking it into using Groq as the backend API to run the large model.

In [31]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    temperature=0, 
    model_name=MODEL,
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("Groq_API_KEY")
)

- now if we try to retrive that who is 'John Adams' it will search him through 'retriever' and grab specific documents where Jhon Adams word is mentioned

In [38]:
retriever.invoke("who is John_Adams?")

[Document(id='ea35fa0f-707a-4567-a365-cf19d8509dd6', metadata={'source': 'knowledge-base\\text_data\\S08_set3_a1.txt.clean', 'source_file': 'S08_set3_a1.txt.clean', 'source_type': 'clean'}, page_content='Birthplace of John Adams, Quincy, Massachusetts.'),
 Document(id='2e835e6d-90d7-4503-8e87-7607c5c85464', metadata={'source_type': 'clean', 'source': 'knowledge-base\\text_data\\S08_set3_a1.txt.clean', 'source_file': 'S08_set3_a1.txt.clean'}, page_content='John Adams was the oldest of three brothers, born on October 30, 1735 (October 19, 1735 by the Old Style, Julian calendar), in Braintree, Massachusetts, though in an area which became part of Quincy, Massachusetts in 1792. His birthplace is now part of Adams National Historical Park. His father, a farmer and a Deacon, also named John (1690-1761), was a fourth-generation descendant of Henry Adams, who immigrated from Barton St David, Somerset, England, to Massachusetts Bay Colony in about 1636,'),
 Document(id='3f5baefe-462c-44e1-8d34-

- Writing System Prompt which will Provided to LLM every time

In [33]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant.
You are chatting with a user.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

- Storing History in this List 'rag_history'

In [34]:
rag_history = []

### **Conversational Memory and Query Rewriting**

- Because users ask follow-up questions like "When was he born?", the database won't know who "he" is. This function asks the LLM to look at the chat history (rag_history) and rewrite the question to be standalone, e.g., "When was John Adams born?", so the database can search accurately.

In [35]:
def rewrite_question(question: str, rag_history):
    if not rag_history:
        return question

    last_turn = rag_history[-1]
    last_user = last_turn["user"]
    last_assistant = last_turn["assistant"]

    rewrite_prompt = f"""
You are given a conversation and a follow-up question.
Rewrite the follow-up question so it is a standalone question.

Conversation:
User: {last_user}
Assistant: {last_assistant}

Follow-up question:
{question}

Standalone question:
"""
    return llm.invoke(rewrite_prompt).content.strip()


### **The Core RAG Engine**

- This is the brain of the app:
   1. It takes the user's question and rewrites it.
   2. It searches the database (retriever.invoke) for chunks of text matching the standalone question.
   3. It pastes those chunks into the SYSTEM_PROMPT_TEMPLATE.
   4. It feeds this massive prompt (the question + the retrieved context) to the Groq LLM.
   5. It saves the answer to memory and returns the text.

In [36]:
def answer_question(question: str, history):
    # history here is Gradio's history — IGNORE IT

    standalone_question = rewrite_question(question, rag_history)

    docs = retriever.invoke(standalone_question)

    if not docs:
        response_text = "I couldn't find relevant information."
        rag_history.append({
            "user": question,
            "assistant": response_text
        })
        return response_text

    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)

    response_text = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=standalone_question)
    ]).content

    # Save ONLY for RAG memory
    rag_history.append({
        "user": question,
        "assistant": response_text
    })

    # ✅ Return ONLY STRING
    return response_text


### **User Interface (Gradio)**

- Instead of forcing to use the command line, this creates a beautiful, interactive Chat UI in our browser where we can talk to our RAG system directly.

In [37]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
